# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a FAIR<sup>2</sup> dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show basic info
print(f"Dataset: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields and their `@id`s.

In [ ]:
# List all record sets and their field @ids
print("Record Sets in Dataset:")
for rs in dataset.metadata.record_sets:
    print(f"- Record Set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '')}")
    if 'fields' in rs:
        print(f"  Fields:")
        for field in rs['fields']:
            print(f"    - Field @id: {field['@id']}, name: {field.get('name','')}, type: {field.get('dataType','')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Choose one or more record sets based on the previous overview.
# For this FAIR^2 dataset, there is usually a main protein quantitation table.
# We'll extract all record set @ids into record_sets_ids.
record_sets_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

# We'll create a dict of DataFrames for each record set for flexible analysis.
all_dataframes = {}
for record_set_id in record_sets_ids:
    print(f"\nExtracting records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        all_dataframes[record_set_id] = pd.DataFrame(records)
        print(f"  Columns: {list(all_dataframes[record_set_id].columns)}")
        print(all_dataframes[record_set_id].head())
    else:
        print("  No records found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Examples include removing outliers, transforming data distributions, and grouping data for summary statistics.

In [ ]:
# For demonstration, select the first record set and one numeric field for EDA based on its @id.
if len(all_dataframes) > 0:
    first_record_set_id = record_sets_ids[0]
    df = all_dataframes[first_record_set_id]

    # Try to find a numeric field (commonly 'coverage', 'peptides', or 'MW' in proteomics)
    numeric_candidates = [c for c in df.columns if (df[c].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[c]))]
    if not numeric_candidates:
        # Fall back on columns that look numeric by their name
        for c in df.columns:
            if any(word in c.lower() for word in ["coverage", "abundance", "mw", "peptides", "count", "quant"]):
                try:
                    df[c] = pd.to_numeric(df[c], errors='coerce')
                    if pd.api.types.is_numeric_dtype(df[c]):
                        numeric_candidates.append(c)
                except:
                    continue
    if numeric_candidates:
        numeric_field = numeric_candidates[0]

        print(f"Using numeric field: {numeric_field}")

        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > mean ({threshold:.3f}): {len(filtered_df)} records")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by another field if available (e.g. protein type/class)
        group_candidates = [c for c in df.columns if c != numeric_field and df[c].dtype == 'object']
        if group_candidates:
            group_field = group_candidates[0]
            print(f"Grouping by: {group_field}")
            grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped.head())
        else:
            print('No groupable fields found for grouping.')
    else:
        print("No obvious numeric fields found for EDA.")
else:
    print("No dataframes were extracted.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(all_dataframes) > 0 and numeric_candidates:
    # Plot histogram of the selected numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If a group field is found, plot boxplot
    if group_candidates:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Insufficient data for visualization.')

## 6. Conclusion
This notebook demonstrated loading a FAIR<sup>2</sup> protein dataset described in Croissant schema, exploring its structure, loading tabular protein quantitation data, performing basic numeric data EDA and normalization, and visualizing value distributions. For in-depth biological or statistical analysis, refer to full field definitions by `@id` and the FAIR^2 metadata specification.